# Notebook 2: Feature Engineering

In this notebook, we will go through feature engineer for both traditional (non-GPS) and GPS features. It necessary that we should have the target table created from Notebook 1. If not, please go back and run Notebook 1 before going over this one.

## 0. Environment Setup

In [41]:
import pandas as pd
import numpy as np
from pathlib import Path

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading

In [42]:
# Read in all dataframes
starter_target = pd.read_parquet("../data/processed/starter_target.parquet")
pps_raw = pd.read_parquet("../data/processed/pps.parquet")
gps_pp_raw = pd.read_parquet("../data/processed/gps_pp.parquet")

In [43]:
# First 10 rows of starter_target
starter_target.head(10)

,starter_id,target_race_id,track_id,race_date,race_number,distance,distance_id,surface,race_type,purse,registration_number,horse_name,post_position,official_position,field_size,finish_score,is_winner,is_top3,split
0,LRL_2025-12-12_1_22010064,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,22010064,Buckin' Right,1,2,8,0.857143,0,1,train
1,LRL_2025-12-12_1_22011846,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,22011846,Rehoboth Avenue,2,4,8,0.571429,0,0,train
2,LRL_2025-12-12_1_21017739,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,21017739,Sippin' Time,3,8,8,0.000000,0,0,train
3,LRL_2025-12-12_1_19002714,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,19002714,Mischief Motion,4,7,8,0.142857,0,0,train
4,LRL_2025-12-12_1_22004844,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,22004844,Over My Cents,5,6,8,0.285714,0,0,train
5,LRL_2025-12-12_1_22002156,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,22002156,Mun Mun Can Run,6,3,8,0.714286,0,1,train
6,LRL_2025-12-12_1_22004645,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,22004645,Bourbon N Lace,7,1,8,1.000000,1,1,train
7,LRL_2025-12-12_1_21003278,LRL_2025-12-12_1,LRL,2025-12-12,1,7F,7.0,D,SOC,27900,21003278,Weekend Wife,8,5,8,0.428571,0,0,train
8,LRL_2025-12-12_2_20020224,LRL_2025-12-12_2,LRL,2025-12-12,2,1 1/16M,8.5,D,SOC,40885,20020224,Mosler Time,1,3,5,0.500000,0,1,train
9,LRL_2025-12-12_2_20016574,LRL_2025-12-12_2,LRL,2025-12-12,2,1 1/16M,8.5,D,SOC,40885,20016574,Feeling Woozy,2,1,5,1.000000,1,1,train


In [44]:
# First 10 rows of pps_raw
pps_raw.head(10)

,has_gps_data,registration_number,horse_name,pp_track,pp_race_date,pp_race_number,pp_country,race_type,grade,distance_id,about_distance_indicator,distance,surface,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,purse,pp_id
0,,22017307,A Cozy Thing,MED,2025-10-10,4,USA,AOC,,5.0,,5F,T,11,9,9,0,0,5,3,810,780,0,0,750,325,100,200,0,0,10,100,144,11,53550,22017307_MED_2025-10-10_4
1,,22017307,A Cozy Thing,CT,2025-10-30,1,USA,CLM,,6.5,,6 1/2F,D,4,6,5,0,0,4,3,770,700,0,0,650,460,300,200,0,0,300,50,27,7,21500,22017307_CT_2025-10-30_1
2,X,22017307,A Cozy Thing,LRL,2025-11-28,2,USA,SOC,,8.0,,1M,D,4,5,7,6,0,5,4,170,420,570,0,810,575,100,200,150,0,400,500,208,8,25740,22017307_LRL_2025-11-28_2
3,X,21003092,A P M Notion,LRL,2025-10-25,7,USA,CLM,,6.0,,6F,D,6,6,4,0,0,4,4,520,410,0,0,550,975,150,100,0,0,250,50,84,8,25700,21003092_LRL_2025-10-25_7
4,X,21003092,A P M Notion,LRL,2025-11-15,10,USA,CLM,,5.5,,5 1/2F,D,4,6,6,0,0,7,6,560,800,0,0,1100,1525,250,150,0,0,0,50,181,7,23220,21003092_LRL_2025-11-15_10
5,X,21003092,A P M Notion,LRL,2025-12-07,1,USA,CLM,,6.0,,6F,D,8,6,6,0,0,5,4,460,1160,0,0,850,625,10,50,0,0,200,100,830,8,23390,21003092_LRL_2025-12-07_1
6,X,21003092,A P M Notion,LRL,2025-12-28,6,USA,CLM,,7.0,,7F,D,5,3,2,0,0,4,8,300,100,0,0,550,2225,10,100,0,0,150,275,606,9,24550,21003092_LRL_2025-12-28_6
7,X,21003092,A P M Notion,LRL,2026-01-09,5,USA,CLM,,6.0,,6F,D,7,6,9,0,0,9,9,510,870,0,0,1460,3325,10,0,0,0,0,0,961,9,19910,21003092_LRL_2026-01-09_5
8,X,23002277,Above the Norm,GP,2025-11-01,5,USA,MSW,,8.0,,1M,D,7,3,4,7,0,8,8,150,200,1000,0,1660,4725,10,50,150,0,0,0,66,8,70000,23002277_GP_2025-11-01_5
9,X,23002277,Above the Norm,LRL,2025-12-21,3,USA,MCL,,6.0,,6F,D,1,1,1,0,0,3,3,0,0,0,0,350,700,300,10,0,0,150,175,7,9,28400,23002277_LRL_2025-12-21_3


In [45]:
# First 10 rows of gps_pp_raw
gps_pp_raw.head(10)

,horse_name,registration_number,pp_track,pp_race_date,pp_race_number,race_type,grade,distance_id,about_distance_indicator,published_value,surface,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,purse,field_size,pp_id
0,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,0.0,4,4,7.01,100.562,1.022,13.3,100.6,1612.5,15.0,224.8,25740,8,22017307_LRL_2025-11-28_2
1,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,0.5,4,4,6.83,93.551,1.151,15.9,100.6,1511.9,14.7,209.8,25740,8,22017307_LRL_2025-11-28_2
2,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,1.0,4,4,6.72,86.728,1.198,17.2,100.6,1411.3,14.4,195.1,25740,8,22017307_LRL_2025-11-28_2
3,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,1.5,5,4,6.69,80.008,1.047,15.4,100.6,1310.7,14.4,180.7,25740,8,22017307_LRL_2025-11-28_2
4,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,2.0,6,4,6.68,73.313,0.863,12.9,101.1,1210.1,14.2,166.3,25740,8,22017307_LRL_2025-11-28_2
5,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,2.5,6,4,6.61,66.634,0.739,11.1,101.2,1109.0,14.3,152.1,25740,8,22017307_LRL_2025-11-28_2
6,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,3.0,6,4,6.50,60.030,0.561,8.8,101.4,1007.8,14.3,137.8,25740,8,22017307_LRL_2025-11-28_2
7,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,3.5,7,4,6.34,53.526,0.442,7.0,101.2,906.4,13.9,123.5,25740,8,22017307_LRL_2025-11-28_2
8,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,4.0,4,4,6.17,47.186,0.356,5.9,101.0,805.2,13.4,109.6,25740,8,22017307_LRL_2025-11-28_2
9,A Cozy Thing,22017307,LRL,2025-11-28,2,SOC,,8.0,,1M,D,4,4.5,4,4,5.83,41.019,0.246,4.2,100.6,704.2,12.9,96.2,25740,8,22017307_LRL_2025-11-28_2


## 2. Traditional (Non-GPS) Feature Engineering

The goal of this section is to build horse-level pre-race eatures using only the `pps_raw` dataframe, which is the `Starters pps` tab in the original Excel sheet.

These are the traditional race metrics we will compare against the GPS ones:

* point-of-call positions
* lengths behind / ahead
* field size
* purse
* odds
* prior finish positions

### i. Clean Special Values

The README doc says that `9999` means the horse did not finish for some length fields. We will convert those to missing values before aggregating, so that the number won't skew our average.

In [22]:
length_cols = [c for c in pps_raw.columns if "length_" in c.lower()]

for c in length_cols:
    pps_raw[c] = pps_raw[c].replace(9999, np.nan)

### ii. Attach Target-Race Date to Each Starter PP

Noted from the previous notebook that some horses might have more than 3 PPs if they compete in multiple Laurel Park races. We only want prior races (at most 3) that happened before the target Laurel race date for data leakage prevention.

In [26]:
starter_key = starter_target[["target_race_id", "registration_number", "race_date"]]

pps_joined = starter_key.merge(pps_raw, on="registration_number", how="left")
pps_joined = pps_joined[pps_joined['pp_race_date'] < pps_joined['race_date']].copy()

pps_joined = pps_joined.sort_values(
    ['target_race_id', 'registration_number', 'pp_race_date'],
    ascending=[True, True, False]
)

# Keep only the 3 most recent prior PPs relative to each target race
pps_joined['pp_rank_recency'] = (
    pps_joined.groupby(['target_race_id', 'registration_number']).cumcount() + 1
)

pps_joined = pps_joined[pps_joined['pp_rank_recency'] <= 3].copy()
pps_joined.head(10)

,target_race_id,registration_number,race_date,has_gps_data,horse_name,pp_track,pp_race_date,pp_race_number,pp_country,race_type,grade,distance_id,about_distance_indicator,distance,surface,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,purse,pp_id,pp_rank_recency
14,LRL_2025-12-12_1,19002714,2025-12-12,X,Mischief Motion,LRL,2025-11-01,1.0,USA,SOC,,8.0,,1M,T,8.0,1.0,1.0,2.0,0.0,4.0,8.0,0.0,0.0,10.0,0.0,310.0,1020.0,250.0,200.0,200.0,0.0,150.0,425.0,677.0,9.0,31950.0,19002714_LRL_2025-11-01_1,1
13,LRL_2025-12-12_1,19002714,2025-12-12,X,Mischief Motion,LRL,2025-09-26,1.0,USA,SOC,,5.5,,5 1/2F,T,6.0,8.0,8.0,0.0,0.0,6.0,7.0,870.0,760.0,0.0,0.0,960.0,950.0,200.0,200.0,0.0,0.0,50.0,275.0,347.0,9.0,31950.0,19002714_LRL_2025-09-26_1,2
12,LRL_2025-12-12_1,19002714,2025-12-12,X,Mischief Motion,LRL,2025-09-06,2.0,USA,ALW,,8.5,,1 1/16M,D,3.0,3.0,5.0,5.0,0.0,5.0,5.0,250.0,360.0,1210.0,0.0,2400.0,3925.0,10.0,0.0,0.0,0.0,0.0,0.0,217.0,5.0,49000.0,19002714_LRL_2025-09-06_2,3
30,LRL_2025-12-12_1,21003278,2025-12-12,X,Weekend Wife,LRL,2025-11-14,9.0,USA,ALW,,6.0,,6F,D,4.0,6.0,6.0,0.0,0.0,6.0,7.0,610.0,510.0,0.0,0.0,910.0,1025.0,600.0,700.0,0.0,0.0,300.0,0.0,297.0,7.0,48500.0,21003278_LRL_2025-11-14_9,1
29,LRL_2025-12-12_1,21003278,2025-12-12,X,Weekend Wife,LRL,2025-09-19,7.0,USA,ALW,,6.0,,6F,D,1.0,7.0,8.0,0.0,0.0,8.0,7.0,550.0,1010.0,0.0,0.0,1450.0,1575.0,10.0,0.0,0.0,0.0,0.0,75.0,96.0,8.0,49000.0,21003278_LRL_2025-09-19_7,2
28,LRL_2025-12-12_1,21003278,2025-12-12,X,Weekend Wife,LRL,2025-04-27,7.0,USA,ALW,,8.0,,1M,D,3.0,3.0,3.0,3.0,0.0,4.0,5.0,150.0,200.0,350.0,0.0,650.0,935.0,50.0,50.0,50.0,0.0,10.0,75.0,34.0,7.0,48300.0,21003278_LRL_2025-04-27_7,3
11,LRL_2025-12-12_1,21017739,2025-12-12,X,Sippin' Time,LRL,2025-11-07,2.0,USA,SOC,,6.0,,6F,D,6.0,3.0,6.0,0.0,0.0,6.0,5.0,360.0,700.0,0.0,0.0,1250.0,1475.0,100.0,0.0,0.0,0.0,0.0,250.0,1325.0,7.0,30370.0,21017739_LRL_2025-11-07_2,1
10,LRL_2025-12-12_1,21017739,2025-12-12,X,Sippin' Time,LRL,2025-10-25,7.0,USA,CLM,,6.0,,6F,D,5.0,7.0,8.0,0.0,0.0,8.0,6.0,670.0,720.0,0.0,0.0,1060.0,1500.0,50.0,0.0,0.0,0.0,0.0,50.0,201.0,8.0,25700.0,21017739_LRL_2025-10-25_7,2
9,LRL_2025-12-12_1,21017739,2025-12-12,,Sippin' Time,DEL,2025-07-05,4.0,USA,SOC,,6.0,,6F,D,6.0,5.0,7.0,0.0,0.0,7.0,7.0,300.0,560.0,0.0,0.0,700.0,735.0,50.0,0.0,0.0,0.0,0.0,0.0,832.0,7.0,25000.0,21017739_DEL_2025-07-05_4,3
22,LRL_2025-12-12_1,22002156,2025-12-12,X,Mun Mun Can Run,LRL,2025-11-22,1.0,USA,SOC,,5.5,,5 1/2F,T,9.0,6.0,9.0,0.0,0.0,7.0,7.0,360.0,770.0,0.0,0.0,910.0,675.0,50.0,0.0,0.0,0.0,150.0,100.0,40.0,9.0,28040.0,22002156_LRL_2025-11-22_1,1


Now, we are ensured we have a copy of 3 recent most PPs for each `target_race_id`. This helps with standardization across all races, ensure that there aren't cases where we have too much data that possible skew the model.

### iii. Engineer Traditional Summary Features

For feature engineering for traditional (non-GPS) data, we will use a mix of:

* recent-race values (last 1)
* average across last 3
* trends across recent races
* similarity to today's race

In [27]:
def add_similarity_flag(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["same_surface"] = (df["surface"] == df["surface_target"]).astype(int)
    df["same_distance"] = (df["distance"] == df["distance_target"]).astype(int)
    return df

In [30]:
# Bring in current race context for similarity features
context = starter_target[["target_race_id", "registration_number", "surface", "distance"]].rename(
    columns={"surface": "surface_target", "distance": "distance_target"}
)

pps_joined = pps_joined.merge(context, on=["target_race_id", "registration_number"], how="left")
pps_joined = add_similarity_flag(pps_joined)
pps_joined.head(10)

,target_race_id,registration_number,race_date,has_gps_data,horse_name,pp_track,pp_race_date,pp_race_number,pp_country,race_type,grade,distance_id,about_distance_indicator,distance,surface,post_position,position_at_point_of_call_1,position_at_point_of_call_2,position_at_point_of_call_3,position_at_point_of_call_4,position_at_point_of_call_5,official_position,length_behind_at_poc_1,length_behind_at_poc_2,length_behind_at_poc_3,length_behind_at_poc_4,length_behind_at_poc_5,length_behind_at_finish,length_ahead_at_poc_1,length_ahead_at_poc_2,length_ahead_at_poc_3,length_ahead_at_poc_4,length_ahead_at_poc_5,length_ahead_at_finish,post_time_odds,field_size,purse,pp_id,pp_rank_recency,surface_target,distance_target,same_surface,same_distance
0,LRL_2025-12-12_1,19002714,2025-12-12,X,Mischief Motion,LRL,2025-11-01,1.0,USA,SOC,,8.0,,1M,T,8.0,1.0,1.0,2.0,0.0,4.0,8.0,0.0,0.0,10.0,0.0,310.0,1020.0,250.0,200.0,200.0,0.0,150.0,425.0,677.0,9.0,31950.0,19002714_LRL_2025-11-01_1,1,D,7F,0,0
1,LRL_2025-12-12_1,19002714,2025-12-12,X,Mischief Motion,LRL,2025-09-26,1.0,USA,SOC,,5.5,,5 1/2F,T,6.0,8.0,8.0,0.0,0.0,6.0,7.0,870.0,760.0,0.0,0.0,960.0,950.0,200.0,200.0,0.0,0.0,50.0,275.0,347.0,9.0,31950.0,19002714_LRL_2025-09-26_1,2,D,7F,0,0
2,LRL_2025-12-12_1,19002714,2025-12-12,X,Mischief Motion,LRL,2025-09-06,2.0,USA,ALW,,8.5,,1 1/16M,D,3.0,3.0,5.0,5.0,0.0,5.0,5.0,250.0,360.0,1210.0,0.0,2400.0,3925.0,10.0,0.0,0.0,0.0,0.0,0.0,217.0,5.0,49000.0,19002714_LRL_2025-09-06_2,3,D,7F,1,0
3,LRL_2025-12-12_1,21003278,2025-12-12,X,Weekend Wife,LRL,2025-11-14,9.0,USA,ALW,,6.0,,6F,D,4.0,6.0,6.0,0.0,0.0,6.0,7.0,610.0,510.0,0.0,0.0,910.0,1025.0,600.0,700.0,0.0,0.0,300.0,0.0,297.0,7.0,48500.0,21003278_LRL_2025-11-14_9,1,D,7F,1,0
4,LRL_2025-12-12_1,21003278,2025-12-12,X,Weekend Wife,LRL,2025-09-19,7.0,USA,ALW,,6.0,,6F,D,1.0,7.0,8.0,0.0,0.0,8.0,7.0,550.0,1010.0,0.0,0.0,1450.0,1575.0,10.0,0.0,0.0,0.0,0.0,75.0,96.0,8.0,49000.0,21003278_LRL_2025-09-19_7,2,D,7F,1,0
5,LRL_2025-12-12_1,21003278,2025-12-12,X,Weekend Wife,LRL,2025-04-27,7.0,USA,ALW,,8.0,,1M,D,3.0,3.0,3.0,3.0,0.0,4.0,5.0,150.0,200.0,350.0,0.0,650.0,935.0,50.0,50.0,50.0,0.0,10.0,75.0,34.0,7.0,48300.0,21003278_LRL_2025-04-27_7,3,D,7F,1,0
6,LRL_2025-12-12_1,21017739,2025-12-12,X,Sippin' Time,LRL,2025-11-07,2.0,USA,SOC,,6.0,,6F,D,6.0,3.0,6.0,0.0,0.0,6.0,5.0,360.0,700.0,0.0,0.0,1250.0,1475.0,100.0,0.0,0.0,0.0,0.0,250.0,1325.0,7.0,30370.0,21017739_LRL_2025-11-07_2,1,D,7F,1,0
7,LRL_2025-12-12_1,21017739,2025-12-12,X,Sippin' Time,LRL,2025-10-25,7.0,USA,CLM,,6.0,,6F,D,5.0,7.0,8.0,0.0,0.0,8.0,6.0,670.0,720.0,0.0,0.0,1060.0,1500.0,50.0,0.0,0.0,0.0,0.0,50.0,201.0,8.0,25700.0,21017739_LRL_2025-10-25_7,2,D,7F,1,0
8,LRL_2025-12-12_1,21017739,2025-12-12,,Sippin' Time,DEL,2025-07-05,4.0,USA,SOC,,6.0,,6F,D,6.0,5.0,7.0,0.0,0.0,7.0,7.0,300.0,560.0,0.0,0.0,700.0,735.0,50.0,0.0,0.0,0.0,0.0,0.0,832.0,7.0,25000.0,21017739_DEL_2025-07-05_4,3,D,7F,1,0
9,LRL_2025-12-12_1,22002156,2025-12-12,X,Mun Mun Can Run,LRL,2025-11-22,1.0,USA,SOC,,5.5,,5 1/2F,T,9.0,6.0,9.0,0.0,0.0,7.0,7.0,360.0,770.0,0.0,0.0,910.0,675.0,50.0,0.0,0.0,0.0,150.0,100.0,40.0,9.0,28040.0,22002156_LRL_2025-11-22_1,1,D,7F,0,0


In [31]:
# Columns to summarize
traditional_numerical_cols = [
    'official_position',
    'post_time_odds',
    'field_size',
    'purse',
    'post_position',
    'position_at_point_of_call_1',
    'position_at_point_of_call_2',
    'position_at_point_of_call_3',
    'position_at_point_of_call_4',
    'position_at_point_of_call_5',
    'length_behind_at_poc_1',
    'length_behind_at_poc_2',
    'length_behind_at_poc_3',
    'length_behind_at_poc_4',
    'length_behind_at_poc_5',
    'length_behind_at_finish',
    'length_ahead_at_poc_1',
    'length_ahead_at_poc_2',
    'length_ahead_at_poc_3',
    'length_ahead_at_poc_4',
    'length_ahead_at_poc_5',
    'length_ahead_at_finish',
    'same_surface',
    'same_distance'
]

existing_traditional_numerical_cols = [c for c in traditional_numerical_cols if c in pps_joined.columns]
existing_traditional_numerical_cols

['official_position',
 'post_time_odds',
 'field_size',
 'purse',
 'post_position',
 'position_at_point_of_call_1',
 'position_at_point_of_call_2',
 'position_at_point_of_call_3',
 'position_at_point_of_call_4',
 'position_at_point_of_call_5',
 'length_behind_at_poc_1',
 'length_behind_at_poc_2',
 'length_behind_at_poc_3',
 'length_behind_at_poc_4',
 'length_behind_at_poc_5',
 'length_behind_at_finish',
 'length_ahead_at_poc_1',
 'length_ahead_at_poc_2',
 'length_ahead_at_poc_3',
 'length_ahead_at_poc_4',
 'length_ahead_at_poc_5',
 'length_ahead_at_finish',
 'same_surface',
 'same_distance']

In [32]:
# Recent-race snapshots
last1 = (
    pps_joined[pps_joined["pp_rank_recency"] == 1]
    [["target_race_id", "registration_number"] + existing_traditional_numerical_cols]
    .copy() 
)
last1 = last1.rename(columns={c: f'last1_{c}' for c in existing_traditional_numerical_cols})

# Average across up to 3 recent PPs
avg3 = (
    pps_joined.groupby(['target_race_id', 'registration_number'])[existing_traditional_numerical_cols]
    .mean()
    .reset_index()
)
avg3 = avg3.rename(columns={c: f'avg3_{c}' for c in existing_traditional_numerical_cols})

# Volatility across recent PPs
std3 = (
    pps_joined.groupby(['target_race_id', 'registration_number'])[existing_traditional_numerical_cols]
    .std()
    .reset_index()
)
std3 = std3.rename(columns={c: f'std3_{c}' for c in existing_traditional_numerical_cols})

In [33]:
# Ading trend features of whether the horse is improving or fading
pivot_finish = (
    pps_joined.pivot_table(
        index=['target_race_id', 'registration_number'],
        columns='pp_rank_recency',
        values='official_position',
        aggfunc='first'
    )
    .reset_index()
    .rename(columns={1: 'finish_last1', 2: 'finish_last2', 3: 'finish_last3'})
)

pivot_finish['trend_finish_last1_minus_last2'] = pivot_finish['finish_last1'] - pivot_finish['finish_last2']
pivot_finish['trend_finish_last2_minus_last3'] = pivot_finish['finish_last2'] - pivot_finish['finish_last3']

In [34]:
# Merge all traditional features
traditional_features = starter_target[['target_race_id', 'registration_number', 'starter_id']].copy()

for piece in [last1, avg3, std3, pivot_finish]:
    traditional_features = traditional_features.merge(
        piece, on=['target_race_id', 'registration_number'], how='left'
    )

# simple availability flags
pp_counts = (
    pps_joined.groupby(['target_race_id', 'registration_number'])
              .size()
              .rename('num_prior_pps')
              .reset_index()
)
traditional_features = traditional_features.merge(pp_counts, on=['target_race_id', 'registration_number'], how='left')
traditional_features['num_prior_pps'] = traditional_features['num_prior_pps'].fillna(0)
traditional_features['has_any_prior_pp'] = (traditional_features['num_prior_pps'] > 0).astype(int)

traditional_features.head(10)

,target_race_id,registration_number,starter_id,last1_official_position,last1_post_time_odds,last1_field_size,last1_purse,last1_post_position,last1_position_at_point_of_call_1,last1_position_at_point_of_call_2,last1_position_at_point_of_call_3,last1_position_at_point_of_call_4,last1_position_at_point_of_call_5,last1_length_behind_at_poc_1,last1_length_behind_at_poc_2,last1_length_behind_at_poc_3,last1_length_behind_at_poc_4,last1_length_behind_at_poc_5,last1_length_behind_at_finish,last1_length_ahead_at_poc_1,last1_length_ahead_at_poc_2,last1_length_ahead_at_poc_3,last1_length_ahead_at_poc_4,last1_length_ahead_at_poc_5,last1_length_ahead_at_finish,last1_same_surface,last1_same_distance,avg3_official_position,avg3_post_time_odds,avg3_field_size,avg3_purse,avg3_post_position,avg3_position_at_point_of_call_1,avg3_position_at_point_of_call_2,avg3_position_at_point_of_call_3,avg3_position_at_point_of_call_4,avg3_position_at_point_of_call_5,avg3_length_behind_at_poc_1,avg3_length_behind_at_poc_2,avg3_length_behind_at_poc_3,avg3_length_behind_at_poc_4,avg3_length_behind_at_poc_5,avg3_length_behind_at_finish,avg3_length_ahead_at_poc_1,avg3_length_ahead_at_poc_2,avg3_length_ahead_at_poc_3,avg3_length_ahead_at_poc_4,avg3_length_ahead_at_poc_5,avg3_length_ahead_at_finish,avg3_same_surface,avg3_same_distance,std3_official_position,std3_post_time_odds,std3_field_size,std3_purse,std3_post_position,std3_position_at_point_of_call_1,std3_position_at_point_of_call_2,std3_position_at_point_of_call_3,std3_position_at_point_of_call_4,std3_position_at_point_of_call_5,std3_length_behind_at_poc_1,std3_length_behind_at_poc_2,std3_length_behind_at_poc_3,std3_length_behind_at_poc_4,std3_length_behind_at_poc_5,std3_length_behind_at_finish,std3_length_ahead_at_poc_1,std3_length_ahead_at_poc_2,std3_length_ahead_at_poc_3,std3_length_ahead_at_poc_4,std3_length_ahead_at_poc_5,std3_length_ahead_at_finish,std3_same_surface,std3_same_distance,finish_last1,finish_last2,finish_last3,trend_finish_last1_minus_last2,trend_finish_last2_minus_last3,num_prior_pps,has_any_prior_pp
0,LRL_2025-12-12_1,22010064,LRL_2025-12-12_1_22010064,9.0,62.0,9.0,28040.0,4.0,9.0,8.0,0.0,0.0,9.0,520.0,720.0,0.0,0.0,1070.0,950.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,6.666667,162.333333,8.666667,31530.000000,4.000000,4.666667,4.333333,3.000000,0.0,6.666667,423.333333,640.000000,240.000000,0.0,703.333333,820.000000,150.000000,100.000000,66.666667,0.0,166.666667,66.666667,0.000000,0.000000,2.081666,91.762374,0.577350,3300.106059,2.000000,3.785939,3.214550,2.645751,0.0,2.081666,150.443788,405.955663,213.775583,0.0,321.455025,233.880311,150.000000,50.000000,76.376262,0.0,144.337567,76.376262,0.00000,0.00000,9.0,6.0,5.0,3.0,1.0,3.0,1
1,LRL_2025-12-12_1,22011846,LRL_2025-12-12_1_22011846,4.0,63.0,12.0,29900.0,5.0,5.0,6.0,6.0,0.0,7.0,1050.0,1510.0,1210.0,0.0,570.0,525.0,100.0,50.0,200.0,0.0,50.0,10.0,0.0,0.0,5.666667,540.666667,9.666667,35533.333333,4.333333,3.000000,3.333333,3.666667,0.0,5.666667,550.000000,753.333333,473.333333,0.0,386.666667,603.333333,150.000000,200.000000,200.000000,0.0,70.000000,20.000000,0.000000,0.000000,2.886751,720.162713,2.081666,6153.318888,1.154701,1.732051,2.309401,2.081666,0.0,2.309401,435.889894,692.844379,637.991640,0.0,159.478316,157.823741,50.000000,180.277564,50.000000,0.0,72.111026,26.457513,0.00000,0.00000,4.0,9.0,4.0,-5.0,5.0,3.0,1
2,LRL_2025-12-12_1,21017739,LRL_2025-12-12_1_21017739,5.0,1325.0,7.0,30370.0,6.0,3.0,6.0,0.0,0.0,6.0,360.0,700.0,0.0,0.0,1250.0,1475.0,100.0,0.0,0.0,0.0,0.0,250.0,1.0,0.0,6.000000,786.000000,7.333333,27023.333333,5.666667,5.000000,7.000000,0.000000,0.0,7.000000,443.333333,660.000000,0.000000,0.0,1003.333333,1236.666667,66.666667,0.000000,0.000000,0.0,0.000000,100.000000,1.000000,0.000000,1.000000,563.410153,0.577350,2919.354952,0.577350,2.000000,1.000000,0.000000,0.0,1.000000,198.578280,87.177979,0.000000,0.0,279.344471,434.635863,28.867513,0.000000,0.000000,0.0,0.000000,132.287566,0.00000,0.00000,5.0,6.0,7.0,-1.0,-1.0,3.0,1
3,LRL_2025-

## 3. GPS Feature Engineering

Here, our goal is to turn the gate-level `GPS PPS` tab (or `gps_pp_raw`) into horse-level pre-race features. The prompt specifically asks whether variables like stride, running time, and distance ran add predictive value over traditional point-of-call metrics.

### i. Keep Only Prior GPS Races

Similar to the traditional PPs, we only use GPS from races that occurred before the target Laurel races

In [46]:
gps_joined = starter_key.merge(gps_pp_raw, on="registration_number", how="left")
gps_joined = gps_joined[gps_joined["pp_race_date"] < gps_joined["race_date"]]

gps_joined = gps_joined.sort_values(
    ['target_race_id', 'registration_number', 'pp_race_date', 'gate'],
    ascending=[True, True, False, False]
)

# Latest 3 GPS-covered prior races per target race / horse
gps_race_rank = (
    gps_joined[['target_race_id', 'registration_number', 'pp_id', 'pp_race_date']]
    .drop_duplicates()
    .sort_values(['target_race_id', 'registration_number', 'pp_race_date'], ascending=[True, True, False])
)
gps_race_rank['gps_pp_rank_recency'] = (
    gps_race_rank.groupby(['target_race_id', 'registration_number']).cumcount() + 1
)

gps_joined = gps_joined.merge(
    gps_race_rank[['target_race_id', 'registration_number', 'pp_id', 'gps_pp_rank_recency']],
    on=['target_race_id', 'registration_number', 'pp_id'],
    how='left'
)

gps_joined = gps_joined[gps_joined['gps_pp_rank_recency'] <= 3].copy()
gps_joined.head(10)

,target_race_id,registration_number,race_date,horse_name,pp_track,pp_race_date,pp_race_number,race_type,grade,distance_id,about_distance_indicator,published_value,surface,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,purse,field_size,pp_id,gps_pp_rank_recency
0,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,7.5,2.0,8.0,6.61,6.615,0.130,2.2,100.7,100.7,16.6,16.6,31950.0,9.0,19002714_LRL_2025-11-01_1,1
1,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,7.0,2.0,8.0,5.51,12.120,0.120,2.1,100.6,201.3,13.8,30.4,31950.0,9.0,19002714_LRL_2025-11-01_1,1
2,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,6.5,2.0,8.0,5.54,17.656,0.106,1.8,101.3,302.6,13.9,44.3,31950.0,9.0,19002714_LRL_2025-11-01_1,1
3,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,6.0,2.0,8.0,6.08,23.740,0.010,0.2,101.3,403.9,14.5,58.8,31950.0,9.0,19002714_LRL_2025-11-01_1,1
4,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,5.5,1.0,8.0,6.48,30.225,0.000,0.0,101.0,504.9,15.2,74.0,31950.0,9.0,19002714_LRL_2025-11-01_1,1
5,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,5.0,1.0,8.0,6.45,36.678,0.000,0.0,100.7,605.6,14.7,88.7,31950.0,9.0,19002714_LRL_2025-11-01_1,1
6,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,4.5,1.0,8.0,6.30,42.975,0.000,0.0,100.6,706.2,14.3,103.0,31950.0,9.0,19002714_LRL_2025-11-01_1,1
7,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,4.0,2.0,8.0,6.27,49.247,0.047,0.8,100.6,806.8,14.5,117.5,31950.0,9.0,19002714_LRL_2025-11-01_1,1
8,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,3.5,2.0,8.0,6.22,55.464,0.085,1.4,100.6,907.4,14.5,132.0,31950.0,9.0,19002714_LRL_2025-11-01_1,1
9,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-11-01,1.0,SOC,,8.0,,1M,T,8.0,3.0,2.0,8.0,6.14,61.608,0.007,0.1,100.7,1008.1,14.5,146.5,31950.0,9.0,19002714_LRL_2025-11-01_1,1


### ii. Engineer Within-Race GPS Summaries

The current raw data is one row per gate. We need one summary per horse per prior race.

In [48]:
gps_joined["section_speed"]    = gps_joined["distance_ran"] / gps_joined["sectional_time"]
gps_joined["stride_length"]    = gps_joined["distance_ran"] / gps_joined["strides"]
gps_joined["stride_frequency"] = gps_joined["strides"] / gps_joined["sectional_time"]

# Gate ordering note: Highest gate is earliest, gate 0 is finish
gps_joined = gps_joined.sort_values(
    ["target_race_id", "registration_number", "pp_id", "gate"],
    ascending=[True, True, True, False]
) 

gps_joined.head(10)

,target_race_id,registration_number,race_date,horse_name,pp_track,pp_race_date,pp_race_number,race_type,grade,distance_id,about_distance_indicator,published_value,surface,post_position,gate,position,official_position,sectional_time,running_time,time_behind,distance_behind,distance_ran,cumulative_distance_ran,strides,cumulative_strides,purse,field_size,pp_id,gps_pp_rank_recency,section_speed,stride_length,stride_frequency
27,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,8.0,3.0,5.0,6.80,6.794,0.303,5.0,100.9,100.9,16.3,16.3,49000.0,5.0,19002714_LRL_2025-09-06_2,3,14.838235,6.190184,2.397059
28,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,7.5,4.0,5.0,6.16,12.952,0.186,3.0,100.8,201.7,14.5,30.8,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.363636,6.951724,2.353896
29,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,7.0,3.0,5.0,6.26,19.210,0.039,0.6,100.8,302.5,15.0,45.8,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.102236,6.720000,2.396166
30,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,6.5,3.0,5.0,6.41,25.614,0.184,2.9,101.3,403.8,15.1,60.9,49000.0,5.0,19002714_LRL_2025-09-06_2,3,15.803432,6.708609,2.355694
31,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,6.0,3.0,5.0,6.37,31.986,0.178,2.9,102.1,505.9,15.1,76.0,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.028257,6.761589,2.370487
32,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,5.5,4.0,5.0,6.19,38.172,0.174,2.8,100.6,606.5,14.8,90.8,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.252019,6.797297,2.390953
33,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,5.0,4.0,5.0,6.14,44.312,0.248,4.1,100.6,707.1,14.4,105.2,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.384365,6.986111,2.345277
34,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,4.5,4.0,5.0,5.97,50.286,0.296,5.0,100.6,807.7,14.3,119.5,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.850921,7.034965,2.395310
35,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,4.0,5.0,5.0,5.95,56.233,0.403,6.9,100.6,908.3,14.3,133.8,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.907563,7.034965,2.403361
36,LRL_2025-12-12_1,19002714,2025-12-12,Mischief Motion,LRL,2025-09-06,2.0,ALW,,8.5,,1 1/16M,D,3.0,3.5,5.0,5.0,6.25,62.482,0.688,11.2,101.7,1010.0,15.0,148.8,49000.0,5.0,19002714_LRL_2025-09-06_2,3,16.272000,6.780000,2.400000


In [49]:
def summarize_one_prior_gps_race(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("gate", ascending=False).copy()
    
    n = len(df)
    first_third = max(1, n // 3)
    last_third  = max(1, n // 3)
    
    early = df.iloc[:first_third]
    late  = df.iloc[-last_third:]
    
    out = {
        'gps_num_gates': n,
        'gps_finish_pos': df['official_position'].iloc[0],
        'gps_avg_section_speed': df['section_speed'].mean(),
        'gps_early_section_speed': early['section_speed'].mean(),
        'gps_late_section_speed': late['section_speed'].mean(),
        'gps_speed_decay': early['section_speed'].mean() - late['section_speed'].mean(),
        'gps_avg_stride_length': df['stride_length'].mean(),
        'gps_avg_stride_freq': df['stride_frequency'].mean(),
        'gps_total_distance_ran': df['distance_ran'].sum(),
        'gps_final_cum_distance': df['cumulative_distance_ran'].max(),
        'gps_avg_time_behind': df['time_behind'].mean(),
        'gps_final_time_behind': df.loc[df['gate'].idxmin(), 'time_behind'],
        'gps_avg_distance_behind': df['distance_behind'].mean(),
        'gps_final_distance_behind': df.loc[df['gate'].idxmin(), 'distance_behind'],
        'gps_best_position': df['position'].min(),
        'gps_worst_position': df['position'].max(),
        'gps_pos_gain': df['position'].iloc[0] - df['position'].iloc[-1],  # start rank - finish checkpoint rank
    }
    return pd.Series(out)

In [50]:
gps_race_features = (
    gps_joined.groupby(['target_race_id', 'registration_number', 'pp_id'])
              .apply(summarize_one_prior_gps_race)
              .reset_index()
)

gps_race_features.head(10)

,target_race_id,registration_number,pp_id,gps_num_gates,gps_finish_pos,gps_avg_section_speed,gps_early_section_speed,gps_late_section_speed,gps_speed_decay,gps_avg_stride_length,gps_avg_stride_freq,gps_total_distance_ran,gps_final_cum_distance,gps_avg_time_behind,gps_final_time_behind,gps_avg_distance_behind,gps_final_distance_behind,gps_best_position,gps_worst_position,gps_pos_gain
0,LRL_2025-12-12_1,19002714,19002714_LRL_2025-09-06_2,17.0,5.0,15.292305,15.827160,13.598078,2.229082,6.549191,2.331273,1715.4,1715.4,1.512471,6.712,22.194118,89.4,3.0,5.0,-2.0
1,LRL_2025-12-12_1,19002714,19002714_LRL_2025-09-26_1,11.0,7.0,17.302048,17.190069,16.715921,0.474148,6.995036,2.473445,1107.1,1107.1,0.994182,1.696,17.172727,26.7,5.0,8.0,-2.0
2,LRL_2025-12-12_1,19002714,19002714_LRL_2025-11-01_1,16.0,8.0,16.196117,16.805002,15.792780,1.012222,6.807145,2.379398,1612.3,1612.3,0.272000,1.803,4.412500,28.2,1.0,8.0,-6.0
3,LRL_2025-12-12_1,21003278,21003278_LRL_2025-04-27_7,16.0,5.0,16.211236,16.955848,15.151763,1.804085,6.711500,2.414008,1613.0,1613.0,0.334313,1.619,4.925000,21.9,1.0,5.0,-4.0
4,LRL_2025-12-12_1,21003278,21003278_LRL_2025-09-19_7,12.0,7.0,16.437004,17.091128,15.540927,1.550201,6.835868,2.405080,1208.5,1208.5,1.108833,2.749,17.858333,41.1,4.0,8.0,-2.0
5,LRL_2025-12-12_1,21003278,21003278_LRL_2025-11-14_9,12.0,7.0,16.517944,17.002250,15.575339,1.426911,6.971675,2.370394,1209.6,1209.6,0.925333,1.799,15.158333,26.8,6.0,7.0,-1.0
6,LRL_2025-12-12_1,21017739,21017739_LRL_2025-10-25_7,12.0,6.0,16.226244,16.930523,15.180053,1.750471,7.042561,2.303378,1210.1,1210.1,1.327500,2.591,21.183333,37.3,6.0,8.0,2.0
7,LRL_2025-12-12_1,21017739,21017739_LRL_2025-11-07_2,12.0,5.0,16.764892,18.188534,14.979502,3.209031,7.018120,2.386057,1218.2,1218.2,0.772417,2.550,11.825000,36.3,1.0,6.0,-4.0
8,LRL_2025-12-12_1,22002156,22002156_LRL_2025-10-10_3,16.0,7.0,16.351334,16.396838,16.382865,0.013973,6.974363,2.347103,1616.5,1616.5,0.553813,0.965,8.956250,14.9,3.0,8.0,-4.0
9,LRL_2025-12-12_1,22002156,22002156_LRL_2025-11-08_6,11.0,1.0,17.407842,17.522700,17.124744,0.397955,7.175572,2.427882,1115.4,1115.4,0.500727,0.000,8.718182,0.0,1.0,5.0,4.0


Now we collapse the latest up-to-3 GPS prior races into one vector per starter

In [51]:
gps_race_features = gps_race_features.merge(
    gps_race_rank[['target_race_id', 'registration_number', 'pp_id', 'gps_pp_rank_recency']],
    on=['target_race_id', 'registration_number', 'pp_id'],
    how='left'
)

gps_num_cols = [
    c for c in gps_race_features.columns
    if c.startswith('gps_') and c not in ['gps_pp_rank_recency']
]

gps_last1 = (
    gps_race_features[gps_race_features['gps_pp_rank_recency'] == 1]
    [['target_race_id', 'registration_number'] + gps_num_cols]
    .rename(columns={c: f'last1_{c}' for c in gps_num_cols})
)

gps_avg3 = (
    gps_race_features.groupby(['target_race_id', 'registration_number'])[gps_num_cols]
    .mean()
    .reset_index()
    .rename(columns={c: f'avg3_{c}' for c in gps_num_cols})
)

gps_std3 = (
    gps_race_features.groupby(['target_race_id', 'registration_number'])[gps_num_cols]
    .std()
    .reset_index()
    .rename(columns={c: f'std3_{c}' for c in gps_num_cols})
)

In [53]:
gps_features = starter_target[['target_race_id', 'registration_number', 'starter_id']].copy()

for piece in [gps_last1, gps_avg3, gps_std3]:
    gps_features = gps_features.merge(
        piece, on=['target_race_id', 'registration_number'], how='left'
    )

gps_counts = (
    gps_race_features.groupby(['target_race_id', 'registration_number'])
                     .size()
                     .rename('num_prior_gps_races')
                     .reset_index()
)
gps_features = gps_features.merge(gps_counts, on=['target_race_id', 'registration_number'], how='left')
gps_features['num_prior_gps_races'] = gps_features['num_prior_gps_races'].fillna(0)
gps_features['has_any_prior_gps'] = (gps_features['num_prior_gps_races'] > 0).astype(int)

gps_features.head(10)

,target_race_id,registration_number,starter_id,last1_gps_num_gates,last1_gps_finish_pos,last1_gps_avg_section_speed,last1_gps_early_section_speed,last1_gps_late_section_speed,last1_gps_speed_decay,last1_gps_avg_stride_length,last1_gps_avg_stride_freq,last1_gps_total_distance_ran,last1_gps_final_cum_distance,last1_gps_avg_time_behind,last1_gps_final_time_behind,last1_gps_avg_distance_behind,last1_gps_final_distance_behind,last1_gps_best_position,last1_gps_worst_position,last1_gps_pos_gain,avg3_gps_num_gates,avg3_gps_finish_pos,avg3_gps_avg_section_speed,avg3_gps_early_section_speed,avg3_gps_late_section_speed,avg3_gps_speed_decay,avg3_gps_avg_stride_length,avg3_gps_avg_stride_freq,avg3_gps_total_distance_ran,avg3_gps_final_cum_distance,avg3_gps_avg_time_behind,avg3_gps_final_time_behind,avg3_gps_avg_distance_behind,avg3_gps_final_distance_behind,avg3_gps_best_position,avg3_gps_worst_position,avg3_gps_pos_gain,std3_gps_num_gates,std3_gps_finish_pos,std3_gps_avg_section_speed,std3_gps_early_section_speed,std3_gps_late_section_speed,std3_gps_speed_decay,std3_gps_avg_stride_length,std3_gps_avg_stride_freq,std3_gps_total_distance_ran,std3_gps_final_cum_distance,std3_gps_avg_time_behind,std3_gps_final_time_behind,std3_gps_avg_distance_behind,std3_gps_final_distance_behind,std3_gps_best_position,std3_gps_worst_position,std3_gps_pos_gain,num_prior_gps_races,has_any_prior_gps
0,LRL_2025-12-12_1,22010064,LRL_2025-12-12_1_22010064,11.0,9.0,17.061265,17.347148,16.147311,1.199837,7.166532,2.381502,1120.3,1120.3,0.972545,1.661,16.472727,25.3,6.0,9.0,-1.0,14.666667,6.666667,16.550100,16.641350,16.293064,0.348286,7.175522,2.307309,1486.266667,1486.266667,0.658848,1.434000,10.915909,22.133333,3.000000,7.333333,-3.000000,3.214550,2.081666,0.449296,0.761638,0.499996,1.193346,0.025391,0.064312,321.767779,321.767779,0.337943,0.431841,5.813998,6.733746,2.645751,1.527525,2.000000,3.0,1
1,LRL_2025-12-12_1,22011846,LRL_2025-12-12_1_22011846,16.0,4.0,16.375796,16.245828,16.274563,-0.028735,6.903549,2.373231,1617.2,1617.2,1.313937,0.909,21.631250,13.7,4.0,7.0,1.0,17.000000,5.666667,16.386257,16.240828,16.342469,-0.101641,6.908212,2.374192,1719.033333,1719.033333,0.669949,1.073000,10.966844,16.633333,2.333333,6.666667,-2.666667,1.000000,2.886751,0.042200,0.290945,0.409885,0.699413,0.014996,0.001763,100.670866,100.670866,0.578956,0.324709,9.571281,5.430776,1.527525,2.516611,4.725816,3.0,1
2,LRL_2025-12-12_1,21017739,LRL_2025-12-12_1_21017739,12.0,5.0,16.764892,18.188534,14.979502,3.209031,7.018120,2.386057,1218.2,1218.2,0.772417,2.550,11.825000,36.3,1.0,6.0,-4.0,12.000000,5.500000,16.495568,17.559529,15.079778,2.479751,7.030340,2.344718,1214.150000,1214.150000,1.049958,2.570500,16.504167,36.800000,3.500000,7.000000,-1.000000,0.000000,0.707107,0.380882,0.889548,0.141810,1.031358,0.017283,0.058463,5.727565,5.727565,0.392503,0.028991,6.617341,0.707107,3.535534,1.414214,4.242641,2.0,1
3,LRL_2025-12-12_1,19002714,LRL_2025-12-12_1_19002714,16.0,8.0,16.196117,16.805002,15.792780,1.012222,6.807145,2.379398,1612.3,1612.3,0.272000,1.803,4.412500,28.2,1.0,8.0,-6.0,14.666667,6.666667,16.263490,16.607410,15.368926,1.238484,6.783791,2.394706,1478.266667,1478.266667,0.926217,3.403667,14.593115,48.100000,3.000000,7.000000,-3.333333,3.214550,1.527525,1.006564,0.702611,1.601554,0.899079,0.223839,0.072312,325.547114,325.547114,0.623022,2.865600,9.167185,35.774712,2.000000,1.732051,2.309401,3.0,1
4,LRL_2025-12-12_1,22004844,LRL_2025-12-12_1_22004844,12.0,1.0,16.528415,17.128222,15.584834,1.543387,6.718628,2.459141,1208.4,1208.4,0.176583,0.000,2.958333,0.0,1.0,3.0,2.0,11.666667,2.333333,16.644545,17.282870,15.415296,1.867575,6.707161,2.480838,1177.633333,1177.633333,0.264891,0.308000,4.396970,4.266667,1.666667,4.333333,1.666667,0.577350,1.154701,0.228375,0.162075,0.441651,0.580250,0.104627,0.019749,58.028815,58.028815,0.078813,0.298953,1.300998,4.067350,0.577350,1.154701,0.577350,3.0,1
5,LRL_2025-12-12_1,22002156,LRL_2025-12-12_1_22002156,11.0,7.0,17.009480,17.205112,16.486927

## 4. Save Output

We now save our traditional features and gps features into processed data for future modeling.

In [56]:
# Save traditional features
traditional_features.to_parquet('../data/feature_engineer/traditional_features.parquet', index=False)
traditional_features.to_csv('../data/feature_engineer/traditional_features.csv', index=False)
print('Saved traditional_features.parquet and .csv')

# Save GPS features
gps_features.to_parquet('../data/feature_engineer/gps_features.parquet', index=False)
gps_features.to_csv('../data/feature_engineer/gps_features.csv', index=False)
print('Saved gps_features.parquet and .csv')

Saved traditional_features.parquet and .csv
Saved gps_features.parquet and .csv
